# Analyses complémentaires et cartographie

On fait les analyses suivantes: 

- les inégalités d'accessibilité pondérées par la population 
- la différence entre densité locale de médecins et accessibilité spatiale 
- les écarts selon la densité territoriale, le niveau de vie, la pauvreté et l'âge 
- les disparités départementales 
- les profils des communes les moins accessibles 
- les cartes statiques 
- une carte interactive destinée au futur site Quarto

In [1]:
pip install folium

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import json
import warnings
import folium
import geopandas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# 1. Chargement de la base finale déjà construite

In [5]:
URL_BASE_ANALYSE = (
    "https://minio.lab.sspcloud.fr/catioui/"
    "projet_DS/processed/base_analyse_finale.parquet"
)

DOSSIER_TABLES = Path("outputs/tables")
DOSSIER_FIGURES = Path("outputs/figures")
DOSSIER_CARTES = Path("outputs/cartes")

for dossier in [
    DOSSIER_TABLES,
    DOSSIER_FIGURES,
    DOSSIER_CARTES,
]:
    dossier.mkdir(parents=True, exist_ok=True)

base = pd.read_parquet(URL_BASE_ANALYSE)

base["CODGEO"] = base["CODGEO"].astype(str)

print("Source :", URL_BASE_ANALYSE)
print("Dimensions :", base.shape)
print("Communes :", base["CODGEO"].nunique())

display(base.head())

Source : https://minio.lab.sspcloud.fr/catioui/projet_DS/processed/base_analyse_finale.parquet
Dimensions : (34858, 57)
Communes : 34858


,CODGEO,LIBGEO,SEXE1_AGEPYR1000,SEXE1_AGEPYR1003,SEXE1_AGEPYR1006,SEXE1_AGEPYR1011,SEXE1_AGEPYR1018,SEXE1_AGEPYR1025,SEXE1_AGEPYR1040,SEXE1_AGEPYR1055,SEXE1_AGEPYR1065,SEXE1_AGEPYR1080,SEXE2_AGEPYR1000,SEXE2_AGEPYR1003,SEXE2_AGEPYR1006,SEXE2_AGEPYR1011,SEXE2_AGEPYR1018,SEXE2_AGEPYR1025,SEXE2_AGEPYR1040,SEXE2_AGEPYR1055,SEXE2_AGEPYR1065,SEXE2_AGEPYR1080,PMUN22,P1,P2,P3,DENS_AAV,LIBDENS_AAV,MED21,TP6021,RD21,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,LIT_SMR,LIT_USLD,NB_SCAN,NB_IRM,NB_MAMM,nb_etablissements,DENS,LIBDENS,DENS7,LIBDENS7,nb_professionnels,population,demande_standardisee,nb_medecins_generalistes,accessibilite_brute,accessibilite_ajustee_age,densite_mg,ratio_age_brute,decile_brute,decile_ajuste_age,population_65_plus,part_65_plus_pct
0,01001,L'Abergement-Clémenciat,19.000,15.000,36.000,42.000,15.000,77.000,87.000,66.000,60.000,20.000,14.000,13.000,20.000,39.000,15.000,74.000,90.000,65.000,68.000,24.000,859.000,0.000,0.000,100.000,4.000,Rural non périurbain,"25,820.000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3,Rural,6,Rural à habitat dispersé,1,859.000,867.711,0,60.836,60.555,0.000,0.995,3,4,172.000,20.023
1,01002,L'Abergement-de-Varey,3.000,6.000,7.000,22.000,6.000,20.000,39.000,12.000,18.000,10.000,3.000,6.000,12.000,12.000,5.000,21.000,35.000,14.000,14.000,8.000,273.000,0.000,0.000,100.000,4.000,Rural non périurbain,"24,480.000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3,Rural,6,Rural à habitat dispersé,1,273.000,270.431,0,108.473,110.057,0.000,1.015,7,7,50.000,18.315
2,01004,Ambérieu-en-Bugey,291.774,327.193,525.371,719.638,733.735,"1,581.246","1,344.792",833.964,888.675,244.548,323.969,283.370,467.591,664.537,736.656,"1,518.093","1,408.495",984.297,"1,151.504",524.551,"15,554.000",0.000,96.934,3.066,2.000,Urbain intermédiaire,"21,660.000",17.000,3.100,31.000,27.000,15.000,73.000,NaN,NaN,NaN,NaN,NaN,3,2,Urbain intermédiaire,2,Centres urbains intermédiaires,521,"15,554.000","14,951.854",26,127.515,130.706,167.160,1.025,8,8,"2,809.278",18.061
3,01005,Ambérieux-en-Dombes,36.262,37.109,65.943,80.082,54.446,193.449,196.441,137.739,121.683,36.337,29.112,33.800,61.125,81.584,53.279,195.597,204.339,131.092,117.452,50.131,"1,917.000",0.000,0.000,100.000,3.000,Rural périurbain,"24,610.000",NaN,2.600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3,Rural,5,Bourgs ruraux,29,"1,917.000","1,861.977",1,88.141,89.490,52.165,1.015,6,6,325.602,16.985
4,01006,Ambléon,2.000,1.000,2.000,1.000,4.000,12.000,11.000,14.000,11.000,2.000,1.000,2.000,2.000,1.000,2.000,10.000,7.000,10.000,15.000,4.000,114.000,0.000,0.000,100.000,4.000,Rural non périurbain,"24,210.000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3,Rural,6,Rural à habitat dispersé,0,114.000,124.706,0,58.393,56.284,0.000,0.964,3,3,32.000,28.070


## 2. Préparation de quelques variables complémentaires

On crée notamment :

- le code département ;
- la variation due à l'ajustement par l'âge ;
- un indicateur `faible_accessibilite` correspondant aux déciles 1 et 2(c'est une convention personnelle du projet mais pas la definition d'un désert médical )

In [7]:
colonnes_num = [
    "population",
    "accessibilite_brute",
    "accessibilite_ajustee_age",
    "densite_mg",
    "nb_medecins_generalistes",
    "MED21",
    "TP6021",
    "RD21",
    "part_65_plus_pct",
    "nb_etablissements",
    "nb_professionnels",
    "LIT_MCO",
    "LIT_SMR",
    "NB_SCAN",
    "NB_IRM",
    "decile_ajuste_age",
]

for col in colonnes_num:
    if col in peuplees.columns:
        if peuplees[col].dtype == "object":
            peuplees[col] = (
                peuplees[col]
                .astype(str)
                .str.replace(",", ".", regex=False)
                .replace({
                    "nan": np.nan,
                    "None": np.nan,
                    "": np.nan,
                })
            )

        peuplees[col] = pd.to_numeric(
            peuplees[col],
            errors="coerce",
        )


def code_departement(codgeo):
    codgeo = str(codgeo)

    if codgeo.startswith(("2A", "2B")):
        return codgeo[:2]

    if codgeo.startswith("97"):
        return codgeo[:3]

    return codgeo[:2]


peuplees["CODE_DEP"] = (
    peuplees["CODGEO"]
    .map(code_departement)
)

peuplees["variation_age_pct"] = np.where(
    peuplees["accessibilite_brute"] > 0,
    100
    * (
        peuplees["accessibilite_ajustee_age"]
        / peuplees["accessibilite_brute"]
        - 1
    ),
    np.nan,
)

if "decile_ajuste_age" in peuplees.columns:
    peuplees["faible_accessibilite"] = (
        peuplees["decile_ajuste_age"]
        .isin([1, 2])
    )

NameError: name 'peuplees' is not defined

In [ ]:
def code_departement(codgeo):
    codgeo = str(codgeo)
    if codgeo.startswith(("2A", "2B")):
        return codgeo[:2]
    if codgeo.startswith("97"):
        return codgeo[:3]
    return codgeo[:2]

peuplees["CODE_DEP"] = peuplees["CODGEO"].map(code_departement)

analyse_departements = (
    resume_par_groupe(
        peuplees,
        "CODE_DEP",
    )
    .sort_values(
        "accessibilite_moyenne_ponderee"
    )
)

display(analyse_departements.head(15))
display(analyse_departements.tail(15))

sauver_table(
    analyse_departements,
    "analyse_departements_complete.csv",
)

In [ ]:
def sauver_table(df, nom, index=False):
    chemin = DOSSIER_TABLES / nom
    df.to_csv(chemin, index=index)
    print("Table enregistrée :", chemin)


def sauver_figure(nom, dpi=180):
    chemin = DOSSIER_FIGURES / nom
    plt.tight_layout()
    plt.savefig(
        chemin,
        dpi=dpi,
        bbox_inches="tight",
    )
    plt.show()
    print("Figure enregistrée :", chemin)


def moyenne_ponderee(valeurs, poids):
    valeurs = pd.to_numeric(
        valeurs,
        errors="coerce",
    )
    poids = pd.to_numeric(
        poids,
        errors="coerce",
    )

    masque = (
        valeurs.notna()
        & poids.notna()
        & (poids > 0)
    )

    if masque.sum() == 0:
        return np.nan

    return np.average(
        valeurs[masque],
        weights=poids[masque],
    )


def quantile_pondere(valeurs, poids, q):
    valeurs = np.asarray(
        valeurs,
        dtype=float,
    )
    poids = np.asarray(
        poids,
        dtype=float,
    )

    masque = (
        np.isfinite(valeurs)
        & np.isfinite(poids)
        & (poids > 0)
    )

    valeurs = valeurs[masque]
    poids = poids[masque]

    if len(valeurs) == 0:
        return np.nan

    ordre = np.argsort(valeurs)

    valeurs = valeurs[ordre]
    poids = poids[ordre]

    cumul = np.cumsum(poids)
    cumul = cumul / cumul[-1]

    return np.interp(
        q,
        cumul,
        valeurs,
    )


def resume_par_groupe(df, groupe):
    lignes = []

    for modalite, g in df.groupby(
        groupe,
        dropna=False,
    ):
        pop = g["population"].fillna(0)

        ligne = {
            groupe: modalite,
            "nb_communes": len(g),
            "population": pop.sum(),
            "accessibilite_moyenne_ponderee":
                moyenne_ponderee(
                    g["accessibilite_ajustee_age"],
                    pop,
                ),
            "accessibilite_mediane_communes":
                g["accessibilite_ajustee_age"].median(),
        }

        if "faible_accessibilite" in g.columns:
            pop_faible = pop[
                g["faible_accessibilite"]
                .fillna(False)
            ].sum()

            ligne[
                "part_population_faible_accessibilite_pct"
            ] = (
                100 * pop_faible / pop.sum()
                if pop.sum() > 0
                else np.nan
            )

        lignes.append(ligne)

    return pd.DataFrame(lignes)

# A. Inégalités nationales d'accessibilité

Les premières statistiques du notebook précédent étaient calculées au niveau des communes.

Ici, on complète avec une lecture **pondérée par la population**, plus proche de la situation vécue par les habitants.

In [ ]:
pop_totale = peuplees["population"].sum()

p10 = quantile_pondere(
    peuplees["accessibilite_ajustee_age"],
    peuplees["population"],
    0.10,
)

p50 = quantile_pondere(
    peuplees["accessibilite_ajustee_age"],
    peuplees["population"],
    0.50,
)

p90 = quantile_pondere(
    peuplees["accessibilite_ajustee_age"],
    peuplees["population"],
    0.90,
)

moyenne_pop = moyenne_ponderee(
    peuplees["accessibilite_ajustee_age"],
    peuplees["population"],
)

part_pop_faible = np.nan

if "faible_accessibilite" in peuplees.columns:
    part_pop_faible = (
        100
        * peuplees.loc[
            peuplees["faible_accessibilite"],
            "population",
        ].sum()
        / pop_totale
    )

kpis_complementaires = pd.DataFrame(
    {
        "indicateur": [
            "Accessibilité moyenne pondérée",
            "P10 pondéré",
            "Médiane pondérée",
            "P90 pondéré",
            "Rapport P90 / P10",
            "Part de population dans les déciles 1-2 (%)",
        ],
        "valeur": [
            moyenne_pop,
            p10,
            p50,
            p90,
            p90 / p10 if p10 > 0 else np.nan,
            part_pop_faible,
        ],
    }
)

display(kpis_complementaires)
sauver_table(
    kpis_complementaires,
    "kpis_complementaires.csv",
)

## Courbe de concentration de l'accessibilité

In [ ]:
tmp = (
    peuplees[
        [
            "accessibilite_ajustee_age",
            "population",
        ]
    ]
    .dropna()
    .sort_values(
        "accessibilite_ajustee_age"
    )
    .copy()
)

tmp["population_cumulee"] = (
    tmp["population"].cumsum()
    / tmp["population"].sum()
)

tmp["accessibilite_totale"] = (
    tmp["population"]
    * tmp["accessibilite_ajustee_age"]
)

tmp["accessibilite_cumulee"] = (
    tmp["accessibilite_totale"].cumsum()
    / tmp["accessibilite_totale"].sum()
)

x = np.r_[0, tmp["population_cumulee"].values]
y = np.r_[0, tmp["accessibilite_cumulee"].values]

indice_inegalite = (
    1 - 2 * np.trapezoid(y, x)
)

print(
    "Indice de concentration de l'accessibilité :",
    round(indice_inegalite, 3),
)

plt.figure(figsize=(7, 7))
plt.plot(
    tmp["population_cumulee"],
    tmp["accessibilite_cumulee"],
    label="Répartition observée",
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Égalité parfaite",
)
plt.xlabel("Part cumulée de la population")
plt.ylabel(
    "Part cumulée de l'accessibilité"
)
plt.title(
    "Concentration de l'accessibilité aux médecins généralistes"
)
plt.legend()
sauver_figure(
    "12_concentration_accessibilite.png"
)

# B. Pourquoi l'indicateur apporte plus qu'une simple densité locale

Une commune peut avoir **zéro médecin sur son territoire** tout en bénéficiant de médecins situés dans les communes voisines.

On compare donc :

- `densite_mg` : médecins présents dans la commune rapportés à sa population ;
- `accessibilite_ajustee_age` : accessibilité spatiale réelle calculée par le projet.

In [ ]:
corr_densite = (
    peuplees[
        [
            "densite_mg",
            "accessibilite_ajustee_age",
        ]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)

print(
    "Corrélation de Spearman densité locale / accessibilité :",
    round(corr_densite, 3),
)

sans_mg = peuplees[
    peuplees["nb_medecins_generalistes"] == 0
].copy()

print(
    "Communes sans MG :",
    len(sans_mg),
)

print(
    "Communes sans MG mais avec accessibilité positive :",
    (
        sans_mg["accessibilite_ajustee_age"]
        > 0
    ).sum(),
)

In [ ]:
x_lim = np.nanpercentile(
    peuplees["densite_mg"],
    98,
)
y_lim = np.nanpercentile(
    peuplees["accessibilite_ajustee_age"],
    98,
)

plt.figure(figsize=(8, 6))
plt.scatter(
    peuplees["densite_mg"],
    peuplees["accessibilite_ajustee_age"],
    alpha=0.25,
    s=10,
)
plt.xlim(0, x_lim)
plt.ylim(0, y_lim)
plt.xlabel(
    "Densité locale de MG pour 100 000 habitants"
)
plt.ylabel(
    "Accessibilité ajustée pour 100 000 habitants"
)
plt.title(
    "Densité locale et accessibilité spatiale"
)
sauver_figure(
    "13_densite_locale_vs_accessibilite.png"
)

In [ ]:
exemples_sans_mg = (
    sans_mg
    .nlargest(
        20,
        "accessibilite_ajustee_age",
    )[
        [
            c for c in [
                "CODGEO",
                "LIBGEO",
                "LIB_DEP",
                "population",
                "densite_mg",
                "accessibilite_ajustee_age",
                "LIBDENS",
            ]
            if c in sans_mg.columns
        ]
    ]
)

display(exemples_sans_mg)

sauver_table(
    exemples_sans_mg,
    "communes_sans_mg_bien_accessibles.csv",
)

# C. Inégalités territoriales approfondies

## 1. Selon la grille de densité INSEE

In [ ]:
if "LIBDENS" in peuplees.columns:
    analyse_densite = (
        resume_par_groupe(
            peuplees,
            "LIBDENS",
        )
        .sort_values(
            "accessibilite_moyenne_ponderee"
        )
    )

    display(analyse_densite)

    sauver_table(
        analyse_densite,
        "analyse_densite_ponderee.csv",
    )

In [ ]:
if "LIBDENS" in peuplees.columns:
    plt.figure(figsize=(10, 5))
    plt.bar(
        analyse_densite["LIBDENS"].astype(str),
        analyse_densite[
            "accessibilite_moyenne_ponderee"
        ],
    )
    plt.xticks(
        rotation=35,
        ha="right",
    )
    plt.ylabel(
        "Accessibilité moyenne pondérée"
    )
    plt.title(
        "Accessibilité selon le type de territoire"
    )
    sauver_figure(
        "14_accessibilite_densite_ponderee.png"
    )

# D. Gradient socio-économique

On répartit les communes en quintiles selon leur niveau de vie et leur taux de pauvreté, puis on calcule l'accessibilité moyenne **pondérée par leur population**.

In [ ]:
def ajouter_quintile(
    df,
    variable,
    nouvelle_colonne,
    sens="croissant",
):
    masque = df[variable].notna()

    labels = [
        "Q1",
        "Q2",
        "Q3",
        "Q4",
        "Q5",
    ]

    df.loc[
        masque,
        nouvelle_colonne,
    ] = pd.qcut(
        df.loc[
            masque,
            variable,
        ].rank(method="first"),
        q=5,
        labels=labels,
    )

    return df

## 2. Niveau de vie médian

In [ ]:
if "MED21" in peuplees.columns:
    peuplees = ajouter_quintile(
        peuplees,
        "MED21",
        "quintile_niveau_vie",
    )

    analyse_revenu = resume_par_groupe(
        peuplees.dropna(
            subset=["quintile_niveau_vie"]
        ),
        "quintile_niveau_vie",
    )

    display(analyse_revenu)

    sauver_table(
        analyse_revenu,
        "accessibilite_quintiles_niveau_vie.csv",
    )

In [ ]:
if "MED21" in peuplees.columns:
    plt.figure(figsize=(8, 5))
    plt.bar(
        analyse_revenu[
            "quintile_niveau_vie"
        ].astype(str),
        analyse_revenu[
            "accessibilite_moyenne_ponderee"
        ],
    )
    plt.xlabel(
        "Quintile de niveau de vie médian"
    )
    plt.ylabel(
        "Accessibilité moyenne pondérée"
    )
    plt.title(
        "Accessibilité et niveau de vie"
    )
    sauver_figure(
        "15_accessibilite_niveau_vie.png"
    )

## 3. Taux de pauvreté

In [ ]:
if "TP6021" in peuplees.columns:
    peuplees = ajouter_quintile(
        peuplees,
        "TP6021",
        "quintile_pauvrete",
    )

    analyse_pauvrete = resume_par_groupe(
        peuplees.dropna(
            subset=["quintile_pauvrete"]
        ),
        "quintile_pauvrete",
    )

    display(analyse_pauvrete)

    sauver_table(
        analyse_pauvrete,
        "accessibilite_quintiles_pauvrete.csv",
    )

In [ ]:
if "TP6021" in peuplees.columns:
    plt.figure(figsize=(8, 5))
    plt.bar(
        analyse_pauvrete[
            "quintile_pauvrete"
        ].astype(str),
        analyse_pauvrete[
            "accessibilite_moyenne_ponderee"
        ],
    )
    plt.xlabel(
        "Quintile de taux de pauvreté"
    )
    plt.ylabel(
        "Accessibilité moyenne pondérée"
    )
    plt.title(
        "Accessibilité et pauvreté"
    )
    sauver_figure(
        "16_accessibilite_pauvrete.png"
    )

# E. Effet de la structure par âge

On examine deux éléments :

1. le niveau d'accessibilité selon la part de personnes âgées ;
2. les communes qui changent le plus de position lorsqu'on passe de l'indicateur brut à l'indicateur ajusté.

In [ ]:
if "part_65_plus_pct" in peuplees.columns:
    peuplees = ajouter_quintile(
        peuplees,
        "part_65_plus_pct",
        "quintile_65_plus",
    )

    analyse_age = resume_par_groupe(
        peuplees.dropna(
            subset=["quintile_65_plus"]
        ),
        "quintile_65_plus",
    )

    display(analyse_age)

    sauver_table(
        analyse_age,
        "accessibilite_quintiles_age.csv",
    )

In [ ]:
if "part_65_plus_pct" in peuplees.columns:
    plt.figure(figsize=(8, 5))
    plt.bar(
        analyse_age[
            "quintile_65_plus"
        ].astype(str),
        analyse_age[
            "accessibilite_moyenne_ponderee"
        ],
    )
    plt.xlabel(
        "Quintile de part des 65 ans ou plus"
    )
    plt.ylabel(
        "Accessibilité moyenne pondérée"
    )
    plt.title(
        "Accessibilité et vieillissement de la population"
    )
    sauver_figure(
        "17_accessibilite_age.png"
    )

In [ ]:
comparaison_age = peuplees[
    [
        c for c in [
            "CODGEO",
            "LIBGEO",
            "LIB_DEP",
            "population",
            "part_65_plus_pct",
            "accessibilite_brute",
            "accessibilite_ajustee_age",
            "variation_age_pct",
        ]
        if c in peuplees.columns
    ]
].dropna(
    subset=[
        "accessibilite_brute",
        "accessibilite_ajustee_age",
    ]
)

plus_penalisees = (
    comparaison_age
    .nsmallest(
        25,
        "variation_age_pct",
    )
)

plus_avantagees = (
    comparaison_age
    .nlargest(
        25,
        "variation_age_pct",
    )
)

print(
    "Communes dont l'ajustement par l'âge réduit le plus l'indicateur"
)
display(plus_penalisees)

print(
    "Communes dont l'ajustement par l'âge augmente le plus l'indicateur"
)
display(plus_avantagees)

sauver_table(
    plus_penalisees,
    "ajustement_age_plus_penalisant.csv",
)
sauver_table(
    plus_avantagees,
    "ajustement_age_plus_favorable.csv",
)

# F. Lien avec l'offre sanitaire plus large

La médecine générale n'est qu'une dimension de l'offre de soins.  
On regarde donc si les communes à forte accessibilité aux généralistes sont aussi mieux dotées en établissements, lits ou équipements.

Il s'agit uniquement d'associations statistiques.

In [ ]:
variables_sanitaires = [
    c for c in [
        "nb_etablissements",
        "nb_professionnels",
        "LIT_MCO",
        "LIT_SMR",
        "LIT_USLD",
        "NB_SCAN",
        "NB_IRM",
        "NB_MAMM",
    ]
    if c in peuplees.columns
]

if variables_sanitaires:
    corr_sante = (
        peuplees[
            variables_sanitaires
            + ["accessibilite_ajustee_age"]
        ]
        .corr(method="spearman")[
            "accessibilite_ajustee_age"
        ]
        .drop(
            "accessibilite_ajustee_age"
        )
        .sort_values()
        .rename("correlation_spearman")
        .reset_index(name="correlation_spearman")
    .rename(columns={"index": "variable"})
    )

    display(corr_sante)

    sauver_table(
        corr_sante,
        "correlations_offre_sanitaire.csv",
    )

In [ ]:
if variables_sanitaires:
    plt.figure(figsize=(8, 5))
    plt.barh(
        corr_sante["variable"],
        corr_sante[
            "correlation_spearman"
        ],
    )
    plt.axvline(0)
    plt.xlabel(
        "Corrélation de Spearman"
    )
    plt.title(
        "Accessibilité MG et autres dimensions de l'offre sanitaire"
    )
    sauver_figure(
        "18_correlations_offre_sanitaire.png"
    )

# G. Disparités départementales

In [ ]:
def code_departement(codgeo):
    codgeo = str(codgeo)
    if codgeo.startswith(("2A", "2B")):
        return codgeo[:2]
    if codgeo.startswith("97"):
        return codgeo[:3]
    return codgeo[:2]

peuplees["CODE_DEP"] = peuplees["CODGEO"].map(code_departement)

analyse_departements = (
    resume_par_groupe(
        peuplees,
        "CODE_DEP",
    )
    .sort_values(
        "accessibilite_moyenne_ponderee"
    )
)

display(analyse_departements.head(15))
display(analyse_departements.tail(15))

sauver_table(
    analyse_departements,
    "analyse_departements_complete.csv",
)

In [ ]:
extremes_dep = pd.concat(
    [
        analyse_departements.head(10),
        analyse_departements.tail(10),
    ],
    ignore_index=True,
)

plt.figure(figsize=(9, 8))
plt.barh(
    extremes_dep["LIB_DEP"].astype(str),
    extremes_dep[
        "accessibilite_moyenne_ponderee"
    ],
)
plt.xlabel(
    "Accessibilité moyenne pondérée"
)
plt.title(
    "Départements aux accessibilités les plus contrastées"
)
sauver_figure(
    "19_departements_extremes.png"
)

# H. Profil des territoires à faible accessibilité

On compare les communes des déciles 1-2 au reste du territoire sur plusieurs caractéristiques.

In [ ]:
if "faible_accessibilite" in peuplees.columns:
    variables_profil = [
        c for c in [
            "population",
            "MED21",
            "TP6021",
            "part_65_plus_pct",
            "nb_medecins_generalistes",
            "nb_etablissements",
            "nb_professionnels",
            "LIT_MCO",
            "NB_IRM",
        ]
        if c in peuplees.columns
    ]

    profil_faible = (
        peuplees
        .groupby(
            "faible_accessibilite"
        )[variables_profil]
        .median()
        .T
        .rename(
            columns={
                False: "deciles_3_10",
                True: "deciles_1_2",
            }
        )
    )

    profil_faible["ecart"] = (
        profil_faible["deciles_1_2"]
        - profil_faible["deciles_3_10"]
    )

    display(profil_faible)

    sauver_table(
        profil_faible.reset_index(
            names="variable"
        ),
        "profil_faible_accessibilite.csv",
    )

# I. Cartographie

Les contours communaux utilisés sont les **Contours administratifs 2025** de data.gouv.fr, en version simplifiée 1000 m.

Cette version est adaptée à une carte nationale et reste suffisamment légère pour le notebook et le futur site Quarto.

In [ ]:
import geopandas as gpd

URL_CONTOURS_COMMUNES = (
    "https://minio.lab.sspcloud.fr/mangane/"
    "Projet_PDS_2026/raw/"
    "communes-1000m-2025-01-08.geojson"
)

contours = gpd.read_file(
    URL_CONTOURS_COMMUNES
)

print(
    "Contours chargés :",
    contours.shape,
)

print(
    "Colonnes :",
    contours.columns.tolist(),
)

In [ ]:
if "code" in contours.columns:
    colonne_code = "code"
elif "CODE" in contours.columns:
    colonne_code = "CODE"
else:
    raise ValueError(
        "Colonne code commune introuvable."
    )


def harmoniser_codgeo_carte(code):
    code = str(code)

    if "75101" <= code <= "75120":
        return "75056"

    if "69381" <= code <= "69389":
        return "69123"

    if "13201" <= code <= "13216":
        return "13055"

    correspondances = {
        "12076": "12218",
        "14011": "14581",
        "49069": "49126",
        "69159": "69114",
    }

    return correspondances.get(
        code,
        code,
    )


contours["CODGEO"] = (
    contours[colonne_code]
    .astype(str)
    .map(harmoniser_codgeo_carte)
)

contours = (
    contours[
        contours["CODGEO"].isin(
            peuplees["CODGEO"]
        )
    ][["CODGEO", "geometry"]]
    .dissolve(
        by="CODGEO",
        as_index=False,
    )
    .to_crs(4326)
)

print(
    "Communes avec géométrie :",
    contours["CODGEO"].nunique(),
)

In [ ]:
colonnes_geo = [
    c for c in [
        "CODGEO",
        "LIBGEO",
        "LIB_DEP",
        "LIBDENS",
        "population",
        "nb_medecins_generalistes",
        "accessibilite_ajustee_age",
        "decile_ajuste_age",
        "MED21",
        "TP6021",
        "part_65_plus_pct",
    ]
    if c in peuplees.columns
]

geo = contours.merge(
    peuplees[colonnes_geo],
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

print("Communes cartographiées :", len(geo))
print(
    "Valeurs d'accessibilité manquantes :",
    geo[
        "accessibilite_ajustee_age"
    ].isna().sum(),
)

## Carte 1 — accessibilité communale en France métropolitaine

In [ ]:
geo_metropole = geo[
    ~geo["CODGEO"].str.startswith("97")
].copy()

ax = geo_metropole.plot(
    column="accessibilite_ajustee_age",
    legend=True,
    figsize=(10, 10),
    linewidth=0.05,
)

ax.set_axis_off()
ax.set_title(
    "Accessibilité ajustée aux médecins généralistes"
)

sauver_figure(
    "20_carte_accessibilite_metropole.png",
    dpi=220,
)

## Carte 2 — communes des deux premiers déciles

In [ ]:
if "decile_ajuste_age" in geo_metropole.columns:
    geo_metropole[
        "classe_accessibilite"
    ] = np.where(
        geo_metropole[
            "decile_ajuste_age"
        ].isin([1, 2]),
        "Déciles 1-2",
        "Déciles 3-10",
    )

    ax = geo_metropole.plot(
        column="classe_accessibilite",
        categorical=True,
        legend=True,
        figsize=(10, 10),
        linewidth=0.05,
    )

    ax.set_axis_off()
    ax.set_title(
        "Communes des deux premiers déciles d'accessibilité"
    )

    sauver_figure(
        "21_carte_deciles_faibles.png",
        dpi=220,
    )

## Carte 3 — accessibilité moyenne par département

In [ ]:
URL_CONTOURS_DEP = (
    "https://contours-administratifs-files.data.gouv.fr/"
    "2025/geojson/departements-1000m.geojson"
)

dep_geo = gpd.read_file(
    URL_CONTOURS_DEP
)

if "code" in dep_geo.columns:
    dep_code = "code"
elif "CODE" in dep_geo.columns:
    dep_code = "CODE"
else:
    raise ValueError(
        "Colonne code département introuvable."
    )

dep_geo["CODE_DEP"] = (
    dep_geo[dep_code].astype(str)
)

dep_geo = dep_geo.merge(
    analyse_departements[
        [
            "CODE_DEP",
            "accessibilite_moyenne_ponderee",
        ]
    ],
    on="CODE_DEP",
    how="left",
)

dep_metropole = dep_geo[
    ~dep_geo["CODE_DEP"].str.startswith("97")
].copy()

ax = dep_metropole.plot(
    column="accessibilite_moyenne_ponderee",
    legend=True,
    figsize=(10, 10),
    linewidth=0.3,
)

ax.set_axis_off()
ax.set_title(
    "Accessibilité moyenne aux médecins généralistes par département"
)

sauver_figure(
    "22_carte_departements.png",
    dpi=220,
)

## Carte 4 — carte interactive communale

La carte est enregistrée en HTML pour pouvoir être intégrée ensuite au site Quarto.

Au survol d'une commune, on affiche notamment :

- le nom de la commune ;
- le département ;
- la population ;
- le nombre de médecins généralistes équivalents ;
- l'accessibilité ajustée ;
- le niveau de vie ;
- le taux de pauvreté ;
- la part des 65 ans ou plus.

In [ ]:
import folium

geo_interactive = geo.copy()

for col in [
    "population",
    "nb_medecins_generalistes",
    "accessibilite_ajustee_age",
    "MED21",
    "TP6021",
    "part_65_plus_pct",
]:
    if col in geo_interactive.columns:
        geo_interactive[col] = (
            pd.to_numeric(
                geo_interactive[col],
                errors="coerce",
            )
            .round(2)
        )

carte = folium.Map(
    location=[46.7, 2.4],
    zoom_start=5,
    control_scale=True,
)

donnees_geojson = json.loads(
    geo_interactive.to_json()
)

couche = folium.Choropleth(
    geo_data=donnees_geojson,
    data=geo_interactive,
    columns=[
        "CODGEO",
        "accessibilite_ajustee_age",
    ],
    key_on="feature.properties.CODGEO",
    legend_name="MG accessibles pour 100 000 habitants",
    highlight=True,
).add_to(carte)

champs = [
    c for c in [
        "LIBGEO",
        "CODE_DEP",
        "LIBDENS",
        "population",
        "nb_medecins_generalistes",
        "accessibilite_ajustee_age",
        "MED21",
        "TP6021",
        "part_65_plus_pct",
    ]
    if c in geo_interactive.columns
]

alias = {
    "LIBGEO": "Commune",
    "CODE_DEP": "Département",
    "LIBDENS": "Type de densité",
    "population": "Population",
    "nb_medecins_generalistes": "MG équivalents",
    "accessibilite_ajustee_age": "Accessibilité ajustée",
    "MED21": "Niveau de vie médian",
    "TP6021": "Taux de pauvreté",
    "part_65_plus_pct": "Part des 65 ans ou plus (%)",
}

couche.geojson.add_child(
    folium.features.GeoJsonTooltip(
        fields=champs,
        aliases=[alias[c] for c in champs],
        sticky=False,
    )
)

chemin_html = (
    DOSSIER_CARTES
    / "carte_accessibilite_interactive.html"
)

carte.save(chemin_html)

print(
    "Carte interactive enregistrée :",
    chemin_html,
)

display(carte)

# J. Fichiers à réutiliser dans Quarto

À la fin de ce notebook, on dispose de :

- tableaux `.csv` dans `outputs/tables/` ;
- graphiques `.png` dans `outputs/figures/` ;
- carte interactive `.html` dans `outputs/cartes/`.

On affiche ci-dessous la liste exacte des fichiers créés.

In [ ]:
print("TABLES")
for f in sorted(
    DOSSIER_TABLES.glob("*")
):
    print("-", f)

print("\nFIGURES")
for f in sorted(
    DOSSIER_FIGURES.glob("*")
):
    print("-", f)

print("\nCARTES")
for f in sorted(
    DOSSIER_CARTES.glob("*")
):
    print("-", f)

# Lecture des résultats pour le Quarto

Le site final pourra s'organiser autour de quatre messages principaux :

1. **L'accessibilité est très différente d'une simple densité locale** : certaines communes sans médecin bénéficient de l'offre des communes voisines.
2. **Les écarts sont territoriaux** : la densité du territoire et le département structurent fortement les niveaux d'accessibilité.
3. **La demande compte autant que l'offre** : le vieillissement et la population environnante modifient l'accessibilité effective.
4. **Les écarts peuvent être croisés avec les caractéristiques socio-économiques** : niveau de vie, pauvreté et offre hospitalière permettent de caractériser les communes les moins bien desservies.

Les tableaux, graphiques et cartes de ce notebook sont conçus pour documenter ces quatre axes sans refaire le calcul de l'indicateur.